In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    sum,
    countDistinct,
    round
)

spark = (
    SparkSession.builder
    .appName("Dashboard Dataset")
    .master("local[*]")
    .getOrCreate()
)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/18 17:42:02 WARN Utils: Your hostname, MacBook-pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.222 instead (on interface en0)
26/06/18 17:42:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/18 17:42:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
# reading Sales Dataset


sales_df = spark.read.parquet(
    "output/gold/sales_fact"
)


In [ ]:

# creating dashboard read dataset based on sales data with the necessary column which can be usefull for visualization

dashboard_df = (
    sales_df
    .groupBy(
        "purchase_year",
        "purchase_month"
    )
    .agg(
        round(
            sum("payment_value"),
            2
        ).alias("revenue"),

        countDistinct(
            "order_id"
        ).alias("orders"),
        countDistinct(
            "customer_id"
        ).alias("customers"),

        countDistinct(
            "product_id"
        ).alias("products")
    )
)

In [4]:
dashboard_df.show()


+-------------+--------------+----------+------+---------+--------+
|purchase_year|purchase_month|   revenue|orders|customers|products|
+-------------+--------------+----------+------+---------+--------+
|         2017|             3| 526961.66|  2641|     2641|    1776|
|         2017|             8|  870105.9|  4293|     4293|    2863|
|         2017|            10|1021169.27|  4568|     4568|    2991|
|         2018|             1|1408365.65|  7220|     7220|    4088|
|         2018|             3|1475599.95|  7188|     7188|    4169|
|         2018|             8|1229643.72|  6452|     6452|    4369|
|         2017|             7| 737293.08|  3969|     3969|    2529|
|         2016|             9|    347.52|     2|        2|       3|
|         2018|             5|1506974.84|  6853|     6853|    4052|
|         2016|            10|  73914.58|   308|      308|     274|
|         2017|            12|1042855.86|  5624|     5624|    3421|
|         2017|             9|1015849.57|  4243|

In [6]:
# saving the dashbaord data set

dashboard_df.write.mode(
    "overwrite"
).parquet(
    "output/dashboard/dashboard_dataset"
)